# TAPNext++ 视频点跟踪

使用项目 `checkpoints/tapnextpp_ckpt.pt` 权重，对自定义视频做在线点跟踪并导出 GIF。

**运行顺序：** 从上到下依次执行。修改 `GPU_ID` 后需 **Restart Kernel**，再从 Cell 1 重跑。

**环境：** 在项目根目录执行 `pip install -e ".[torch]"`（需 `torch`、`torchvision`、`einops`）。无需安装 `tensorflow_datasets`。

## 常用变量说明

各 Code Cell 之间通过下表变量传递状态。**改配置**主要在 Cell 1；**改视频/采样**从 Cell 3 重跑；**改 GIF** 只重跑 Cell 5。

### 配置与路径（Cell 1）

| 变量 | 类型 / 形状 | 含义 |
|------|-------------|------|
| `GPU_ID` | `int` | 物理 GPU 编号（`nvidia-smi` 中可见）。写入 `CUDA_VISIBLE_DEVICES`，改后需 Restart Kernel |
| `device` | `torch.device` | 计算设备，当前为 `cuda`（映射到所选 GPU） |
| `PROJECT_ROOT` | `Path` | 项目根目录（自动根据 `checkpoints/tapnextpp_ckpt.pt` 定位） |
| `CKPT_PATH` | `Path` | TAPNext++ 权重：`checkpoints/tapnextpp_ckpt.pt` |
| `VIDEO_PATH` | `Path` | 输入视频路径 |
| `OUTPUT_GIF` | `Path` | 输出 GIF 路径 |
| `OUTPUT_FPS` | `int` | 导出 GIF 帧率 |
| `IMAGE_SIZE` | `(H, W)` | 模型输入分辨率，默认 `(256, 256)`，需与 checkpoint 一致 |
| `NUM_QUERY_Y`, `NUM_QUERY_X` | `int` | 第一帧网格采样行/列数；总点数 `N = NUM_QUERY_Y × NUM_QUERY_X` |

### 模型（Cell 2）

| 变量 | 类型 | 含义 |
|------|------|------|
| `model` | `TAPNext` | TAPNext++ 模型实例，`.eval()` 推理模式 |
| `ckpt` | `dict` | `torch.load` 的 checkpoint；权重在 `ckpt['state_dict']`，键名带 `tapnext.` 前缀需剥掉 |

### 视频与 query（Cell 3）

| 变量 | 类型 / 形状 | 含义 |
|------|-------------|------|
| `frames` | `np.ndarray` `[T, H, W, 3]` | 解码并 resize 后的 RGB 帧，`uint8`，范围 `[0, 255]`，用于可视化 |
| `T`, `H`, `W` | `int` | 帧数、高、宽 |
| `query_points` | `np.ndarray` `[1, N, 3]` | 查询点，格式 **`(t, y, x)`** 像素坐标；第一帧跟踪时 **`t = 0`** |
| `video` | `torch.Tensor` `[1, T, H, W, 3]` | 模型输入视频，**float**，范围 **`[-1, 1]`**，在 `device` 上 |
| `query_points_tensor` | `torch.Tensor` `[1, N, 3]` | `query_points` 的 GPU 版本 |

### 跟踪结果（Cell 4）

| 变量 | 类型 / 形状 | 含义 |
|------|-------------|------|
| `tracking_state` | `TAPNextTrackingState` | 在线跟踪隐状态；首帧由 `query_points` 初始化，后续帧传入 `state=` |
| `pred_tracks` | `Tensor` `[1, 1, N, 2]` | 当前帧预测位置，**`(x, y)`** 像素坐标 |
| `visible_logits` | `Tensor` `[1, 1, N, 1]` | 可见性 logit；`> 0` 表示可见 |
| `tracks` | `np.ndarray` `[T, N, 2]` | 全序列轨迹，**`(x, y)`** |
| `visible` | `np.ndarray` `[T, N]` 或 `[T, N, 1]` | 每帧每点是否可见（由 `visible_logits > 0` 得到） |
| `use_amp` | `bool` | 是否在 CUDA 上启用 FP16 自动混合精度 |

### 可视化（Cell 5）

| 变量 | 类型 | 含义 |
|------|------|------|
| `vis_frames` | `list` of `[H, W, 3]` | 每帧叠加跟踪点后的 RGB 图像，供 `imageio.mimsave` 写 GIF |

### 坐标约定

- **存储 / 模型 query**：`(t, y, x)`，像素坐标，`t` 为帧索引（首帧查询为 `0`）
- **模型输出 `tracks`**：`(x, y)` 像素坐标（画圆时用 `cv2.circle(img, (x, y), ...)`）
- **OpenCV 画点**：`(x, y)` 为列、行，与 `tracks` 一致

In [15]:
# Cell 1 — 环境与 GPU（必须在 import torch 之前设置）
import os
import sys
import types
from pathlib import Path

GPU_ID = 0  # nvidia-smi 中看到的 GPU 编号，按需修改
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

# 先定位项目根目录，优先使用本地源码（避免 pip 版 tapnet/__init__.py 拉取 TF/JAX）
for candidate in (Path('.').resolve(), Path('..').resolve()):
    if (candidate / 'checkpoints' / 'tapnextpp_ckpt.pt').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        '找不到 checkpoints/tapnextpp_ckpt.pt，'
        '请确认权重已下载到项目根目录的 checkpoints/ 下'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import imageio
import numpy as np
import torch

# tapnet/__init__.py 会 legacy 导入 evaluation_datasets（需要 tensorflow_datasets），
# 与 TAPNext++ 推理无关。预先注册空包以跳过 __init__.py。
if 'tapnet' not in sys.modules:
    _tapnet_pkg = types.ModuleType('tapnet')
    _tapnet_pkg.__path__ = [str(PROJECT_ROOT / 'tapnet')]
    sys.modules['tapnet'] = _tapnet_pkg

from tapnet.tapnext.tapnext_torch import TAPNext

IMAGE_SIZE = (256, 256)
NUM_QUERY_Y, NUM_QUERY_X = 16, 16  # 第一帧网格采样密度

CKPT_PATH = PROJECT_ROOT / 'checkpoints' / 'tapnextpp_ckpt.pt'
VIDEO_PATH = PROJECT_ROOT / 'tap_test_sintel.mp4'  # 换成你的视频路径
OUTPUT_GIF = PROJECT_ROOT / 'use-scripts' / 'tapnextpp_demo.gif'
OUTPUT_FPS = 10

if not torch.cuda.is_available():
    raise RuntimeError(f'CUDA 不可用，请检查 GPU {GPU_ID} 是否空闲且驱动正常')
device = torch.device('cuda')
print(f'使用 GPU: {torch.cuda.get_device_name(0)} (CUDA_VISIBLE_DEVICES={GPU_ID})')
print(f'项目根目录: {PROJECT_ROOT}')
print(f'Checkpoint: {CKPT_PATH}')
print(f'视频: {VIDEO_PATH}')
print(f'输出: {OUTPUT_GIF}')

使用 GPU: NVIDIA GeForce RTX 4090 D (CUDA_VISIBLE_DEVICES=0)
项目根目录: /data03/workspace/tjc/tapnet
Checkpoint: /data03/workspace/tjc/tapnet/checkpoints/tapnextpp_ckpt.pt
视频: /data03/workspace/tjc/tapnet/tap_test_sintel.mp4
输出: /data03/workspace/tjc/tapnet/use-scripts/tapnextpp_demo.gif


## 2. 加载 TAPNext++ 模型

权重较大，只需在每个 kernel 会话中运行一次；改视频或 query 点时不必重跑本格。

In [11]:
model = TAPNext(image_size=IMAGE_SIZE)
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
model.load_state_dict(
    {k.replace('tapnext.', ''): v for k, v in ckpt['state_dict'].items()}
)
model.to(device)
model.eval()
print('TAPNext++ 模型加载完成')

TAPNext++ 模型加载完成


## 3. 读取视频并采样 query points

Query 格式为 `(t, y, x)` 像素坐标，第一帧查询时 `t=0`。

In [16]:
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f'视频不存在: {VIDEO_PATH}')

cap = cv2.VideoCapture(str(VIDEO_PATH))
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, IMAGE_SIZE)
    frames.append(frame)
cap.release()

if len(frames) == 0:
    raise ValueError(f'未能从视频中读取到任何帧: {VIDEO_PATH}')

frames = np.stack(frames, axis=0)  # [T, H, W, 3] uint8
T, H, W, _ = frames.shape

ys = np.linspace(0, H - 1, NUM_QUERY_Y)
xs = np.linspace(0, W - 1, NUM_QUERY_X)
grid_y, grid_x = np.meshgrid(ys, xs, indexing='ij')
query_points = np.stack([
    np.zeros(grid_y.size, dtype=np.float32),
    grid_y.reshape(-1),
    grid_x.reshape(-1),
], axis=-1)[None, ...]  # [1, N, 3]

video = torch.from_numpy(frames).float()[None] / 255.0 * 2.0 - 1.0
video = video.to(device)
query_points_tensor = torch.from_numpy(query_points).float().to(device)

print(f'视频: {T} 帧, {H}x{W}, {query_points.shape[1]} 个 query points')

[mov,mp4,m4a,3gp,3g2,mj2 @ 0x580c5d3928c0] Referenced QT chapter track not found


视频: 240 帧, 256x256, 256 个 query points


## 4. 在线 tracking

逐帧推理，与 `colabs/torch_tapnextpp_demo.ipynb` 一致。

In [17]:
tracks_list = []
visible_list = []
tracking_state = None
use_amp = device.type == 'cuda'

with torch.no_grad(), torch.amp.autocast(
    device.type, dtype=torch.float16, enabled=use_amp
):
    for t in range(T):
        frame_t = video[:, t : t + 1]  # [1, 1, H, W, 3]
        if t == 0:
            pred_tracks, _, visible_logits, tracking_state = model(
                video=frame_t,
                query_points=query_points_tensor,
            )
        else:
            pred_tracks, _, visible_logits, tracking_state = model(
                video=frame_t,
                state=tracking_state,
            )
        tracks_list.append(pred_tracks[0, 0].float().cpu().numpy())
        visible_list.append((visible_logits[0, 0] > 0).float().cpu().numpy())

tracks = np.stack(tracks_list, axis=0)   # [T, N, 2]，(x, y)
visible = np.stack(visible_list, axis=0)  # [T, N]
print(f'跟踪完成: tracks {tracks.shape}, visible {visible.shape}')

跟踪完成: tracks (240, 256, 2), visible (240, 256, 1)


## 5. 可视化并导出 GIF

可反复运行本格以调整 `OUTPUT_FPS` 或绘制样式，无需重跑推理。

In [18]:
vis_frames = []
for t in range(T):
    img = frames[t].copy()
    for i in range(tracks.shape[1]):
        if not visible[t, i]:
            continue
        x, y = tracks[t, i]
        x, y = int(round(x)), int(round(y))
        if 0 <= y < H and 0 <= x < W:
            cv2.circle(img, (x, y), 2, (0, 255, 0), -1)
    vis_frames.append(img)

OUTPUT_GIF.parent.mkdir(parents=True, exist_ok=True)
imageio.mimsave(str(OUTPUT_GIF), vis_frames, fps=OUTPUT_FPS)
print(f'已保存: {OUTPUT_GIF} ({T} 帧, {tracks.shape[1]} 点, {OUTPUT_FPS} fps)')

已保存: /data03/workspace/tjc/tapnet/use-scripts/tapnextpp_demo.gif (240 帧, 256 点, 10 fps)
